# 02. SageMaker Pipeline으로 GR00T Fine-tuning 실행

이 노트북은 GR00T fine-tuning을 SageMaker Pipeline(Transform→Train→SmokeEval→Gate→Register)으로 실행합니다. SmokeGate를 통과한 모델만 Model Package Group에 등록되며, 등록된 모델을 FSx for Lustre로 마운트해 IsaacSim이 로드합니다. 워크숍 가이드 **모듈 4**와 함께 진행합니다.

## 선행 조건

- `setup-notebooks.sh` 실행 완료 (모듈 3.3)
- 커널 = `GR00T (uv)`
- 학습 컨테이너 이미지 빌드 완료 (모듈 3.4)

In [ ]:
from pathlib import Path
import yaml
DOMAIN_ROOT = Path.cwd().parent          # notebooks/ 의 부모 = groot/
CONFIG = yaml.safe_load((DOMAIN_ROOT / "config.yaml").read_text())
aws = CONFIG["aws"]; ecr = CONFIG.get("ecr", {}); model = CONFIG.get("model", {})
train = CONFIG.get("training", {}); mlflow = CONFIG.get("mlflow", {})
BUCKET, REGION, ROLE = aws["bucket_name"], aws.get("region","us-east-1"), aws["role_arn"]
TRAINING_IMAGE_URI = ecr["training_uri"]
assert BUCKET and ROLE and TRAINING_IMAGE_URI, "config.yaml 값 누락 — update-config 먼저 실행"
print(BUCKET, REGION); print(TRAINING_IMAGE_URI)

### (선택) 학습 컨테이너 이미지 재빌드

학습 이미지는 **모듈 3.4에서 이미 빌드했습니다**. Dockerfile 등을 수정해 재빌드가 필요할 때만 아래 셀의 주석을 해제해 실행하세요 (약 20~40분 소요).

데이터셋 업로드는 별도로 하지 않습니다 — 파이프라인의 TransformDataset 스텝이 HF 데이터셋 다운로드/검증/staging을 담당합니다.

In [ ]:
# 학습 이미지는 모듈 3.4에서 이미 빌드했습니다. 재빌드가 필요할 때만 주석을 해제하세요 (약 20~40분).
# !cd {DOMAIN_ROOT} && uv run python training/scripts/trigger_build.py --type training

이 파이프라인은 Transform→Train→SmokeEval→Gate→Register 5단계입니다. TransformDataset이 HF 다운로드·검증·staging을 담당하므로 별도 업로드가 필요 없습니다. SmokeGate는 품질 지표가 아니라 모델이 로드·추론되는지 확인하는 sanity + governance 게이트입니다.

## 1단계: SageMaker 세션

파이프라인 파라미터(EmbodimentTag, HfDatasetId, InstanceType, EvalInstanceType, MaxSteps, GlobalBatchSize, NumGpus)는 `build_pipeline`이 내부적으로 정의합니다.

In [ ]:
import warnings; warnings.filterwarnings("ignore", category=SyntaxWarning)  # sagemaker 의존성의 무해한 경고 숨김
import boto3
from sagemaker.workflow.pipeline_context import PipelineSession
session = PipelineSession(boto_session=boto3.Session(region_name=REGION))

## 2단계: Model Package Group (멱등 생성)

In [ ]:
import boto3
sm = boto3.client("sagemaker", region_name=REGION)
MPG = (model.get("package_group_name", "groot-sm-models") or "groot-sm-models")
if aws.get("alias"):
    MPG = f"{MPG}-{aws['alias']}"
try:
    sm.create_model_package_group(ModelPackageGroupName=MPG,
        ModelPackageGroupDescription="GR00T fine-tuned models (smoke-gated)")
    print("생성:", MPG)
except sm.exceptions.ClientError as e:
    if "already exists" in str(e) or "ValidationException" in str(e):
        print("이미 존재:", MPG)
    else:
        raise

## 3단계: 파이프라인 조립 + 업서트(정의 등록)

`build_pipeline`이 Transform→Train→SmokeEval→Gate→Register 스텝을 모두 구성합니다. `upsert`는 정의를 등록합니다.

In [ ]:
import sys
sys.path.insert(0, str(DOMAIN_ROOT / "pipeline"))
from build_pipeline import build_pipeline
env = {"SM_HP_WANDB_API_KEY": "ssm:/groot/wandb-key"}
if mlflow.get("tracking_server_arn"):
    env.update({"MLFLOW_TRACKING_URI": mlflow["tracking_server_arn"],
                "MLFLOW_EXPERIMENT_NAME": mlflow.get("experiment_name", "groot-sm-finetune"),
                "HF_MLFLOW_LOG_ARTIFACTS": "true"})
pipeline = build_pipeline(
    session=session, role=ROLE, training_image_uri=TRAINING_IMAGE_URI,
    bucket=BUCKET, source_root=str(DOMAIN_ROOT),
    model_prefix=model.get("s3_prefix", "models/groot-sm"),
    model_package_group=MPG,
    hf_dataset_id=CONFIG.get("dataset", {}).get("hf_dataset_id", "LightwheelAI/leisaac-pick-orange"),
    transform_instance_type=CONFIG.get("transform", {}).get("instance_type", "ml.m5.2xlarge"),
    train_instance_type=train.get("instance_type", "ml.g5.12xlarge"),
    eval_instance_type=CONFIG.get("eval", {}).get("instance_type", "ml.g5.2xlarge"),
    max_steps=int(train.get("max_steps", 100)),
    global_batch_size=int(train.get("global_batch_size", 32)),
    save_steps=int(train.get("save_steps", 50)),
    num_gpus=int(train.get("num_gpus", 0) or 0),
    alias=aws.get("alias", "") or "", env=env)
pipeline.upsert(role_arn=ROLE)
print("업서트 완료:", pipeline.name)

## 4단계: 실행

**첫 실행은 Quick validation(100 스텝, 배치 4)으로 시작하세요** — 목적은 좋은 모델이 아니라 다섯 스텝이 끝까지 도는지 확인하는 것입니다. 본격 학습은 아래 셀에서 `MaxSteps=6000`, `GlobalBatchSize=32`로 바꿔 같은 셀을 다시 실행하면 됩니다 (정의 재등록 불필요, 같은 데이터셋이면 TransformDataset은 30일 캐시로 건너뜁니다).

In [ ]:
execution = pipeline.start(parameters={
    "EmbodimentTag": "NEW_EMBODIMENT",
    "HfDatasetId": CONFIG.get("dataset", {}).get("hf_dataset_id", "LightwheelAI/leisaac-pick-orange"),
    "MaxSteps": 100,        # Quick validation — 본격 학습은 6000
    "GlobalBatchSize": 4})  # Quick validation — 본격 학습은 32
print("실행 ARN:", execution.arn)

전체 실행 상태와 스텝별 진행 상황은 아래 두 셀로 조회합니다 (SageMaker Studio의 Pipelines 화면에서도 그래프로 확인 가능 — 워크숍 가이드 모듈 4.4 참고). Quick validation 기준 전체 약 35~45분 소요됩니다.

In [ ]:
execution.describe()["PipelineExecutionStatus"]

In [ ]:
# 스텝별 진행 상황 (TransformDataset → GR00TFinetune → SmokeEval → SmokeGate → RegisterModel)
steps = execution.list_steps()
if not steps:
    print("스텝이 아직 생성되지 않았습니다 — 잠시 후 이 셀을 다시 실행하세요.")
for s in reversed(steps):
    print(f"{s['StepName']:20s} {s['StepStatus']}")

In [ ]:
pkgs = sm.list_model_packages(ModelPackageGroupName=MPG,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=1)["ModelPackageSummaryList"]
if pkgs:
    print("최신 ModelPackage:", pkgs[0]["ModelPackageArn"])
    print("승인 상태:", pkgs[0]["ModelApprovalStatus"])

## 완료 후 안내

- SmokeGate를 통과한 모델은 Model Package Group에 `Approved` 상태로 등록됩니다.
- SageMaker 콘솔 Pipelines 탭에서 실행 상태를, Model Registry 탭에서 등록된 모델 패키지를 확인할 수 있습니다.
- MLflow tracking server 링크에서 학습 메트릭을 확인할 수 있습니다.
- export된 checkpoint는 `s3://<bucket>/models/groot-sm/<execution-id>/`에서 확인합니다 — 모듈 5(`03_closed_loop_eval.ipynb`)에서 `aws s3 sync`로 받아 사용합니다.